# Rupiah Banknotes Dataset — Starter EDA & PyTorch Baseline

This starter notebook demonstrates how to explore and train a baseline classifier on the **Rupiah Banknotes Dataset (Indonesian Currency)** (`1,392` JPG images across Rp1,000 – Rp100,000 plus mixed banknote scenes).

**What this notebook covers:**
1. Inspecting folder structure and class distribution
2. Visualizing sample banknotes from every class
3. Building a PyTorch `Dataset` & `DataLoader` with train/validation split
4. Training a fast **ResNet18** transfer learning baseline

- **Dataset:** [Rupiah Banknotes Dataset on Kaggle](https://www.kaggle.com/datasets/mkaspulanwar/rupiah-banknotes-dataset)
- **Companion GitHub Repo:** [anwarpage/indonesian-rupiah-dataset](https://github.com/anwarpage/indonesian-rupiah-dataset)

In [ ]:
import os
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# Auto-locate dataset directory on Kaggle
CANDIDATES = [
    Path('/kaggle/input/rupiah-banknotes-dataset'),
    Path('/kaggle/input/datasets/mkaspulanwar/rupiah-banknotes-dataset'),
]
DATA_DIR = next((p for p in CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    for p in Path('/kaggle/input').rglob('idr_1000'):
        DATA_DIR = p.parent
        break

assert DATA_DIR is not None, 'Dataset folder not found under /kaggle/input!'
print('Dataset directory:', DATA_DIR)

CLASS_ORDER = [
    'idr_1000', 'idr_2000', 'idr_5000', 'idr_10000',
    'idr_20000', 'idr_50000', 'idr_100000', 'idr_mix'
]
CLASSES = [c for c in CLASS_ORDER if (DATA_DIR / c).is_dir()]
print('Found classes:', CLASSES)

## 1. Class Distribution

In [ ]:
counts = {}
for cls in CLASSES:
    imgs = [f for f in (DATA_DIR / cls).iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    counts[cls] = len(imgs)

total_images = sum(counts.values())
print(f'Total images: {total_images}')
for k, v in counts.items():
    print(f'  {k:12s}: {v:4d} images')

plt.figure(figsize=(9, 4.5))
bars = plt.bar(counts.keys(), counts.values(), color='#2563eb')
plt.title(f'Rupiah Banknotes Class Distribution (Total: {total_images} images)', fontsize=13)
plt.ylabel('Number of Images')
plt.xticks(rotation=25)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 3, str(h), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Sample Images from Every Class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, cls in zip(axes.flat, CLASSES):
    imgs = sorted((DATA_DIR / cls).glob('*.jpg'))
    sample_path = imgs[0]
    with Image.open(sample_path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((512, 512))
        ax.imshow(im)
    ax.set_title(f'{cls} ({counts[cls]} imgs)', fontsize=11)
    ax.axis('off')

plt.suptitle('Sample Indonesian Rupiah Banknotes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. PyTorch Baseline Classifier (7 Single Denominations)

Here we filter to the 7 single-denomination classes (`idr_1000` through `idr_100000`) for clean single-label classification, reserving `idr_mix` for multi-object / object-detection experiments.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split

SINGLE_CLASSES = ['idr_1000', 'idr_2000', 'idr_5000', 'idr_10000', 'idr_20000', 'idr_50000', 'idr_100000']
label2idx = {c: i for i, c in enumerate(SINGLE_CLASSES)}

samples = []
for cls in SINGLE_CLASSES:
    for p in sorted((DATA_DIR / cls).glob('*.jpg')):
        samples.append((str(p), label2idx[cls]))

paths = [s[0] for s in samples]
labels = [s[1] for s in samples]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f'Train: {len(train_paths)} | Val: {len(val_paths)}')

class RupiahDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            im = ImageOps.exif_transpose(im).convert('RGB')
            if self.transform:
                im = self.transform(im)
        return im, self.labels[idx]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(RupiahDataset(train_paths, train_labels, train_tf), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(RupiahDataset(val_paths, val_labels, val_tf), batch_size=32, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Quick offline baseline (works on CPU without internet; switch weights='DEFAULT' if internet is enabled)
try:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    print('Loaded ImageNet pretrained ResNet18 weights.')
except Exception:
    model = models.resnet18(weights=None)
    print('Internet disabled — initialized ResNet18 with random weights (verify pipeline).')

model.fc = nn.Linear(model.fc.in_features, len(SINGLE_CLASSES))
model = model.to(device)

# Run a fast sanity-check forward pass on 1 batch
x_batch, y_batch = next(iter(train_loader))
with torch.no_grad():
    logits = model(x_batch.to(device))
print(f'Sanity check OK! Input batch: {tuple(x_batch.shape)} -> Output logits: {tuple(logits.shape)}')